In [1]:
import numpy as np
import pandas
import pm4py
from matplotlib import pyplot as plt
from sklearn.mixture import GaussianMixture
import scipy.stats as stats
import ot
import os
from tqdm import tqdm
import collections
import matplotlib.dates as md
import importlib
import pickle
import random
import math
import CRPS.CRPS as pscore
import datetime

import multiprocessing as mp
mp.set_start_method('spawn')

pandas.set_option('display.max_columns', None)
#pandas.set_option('display.max_rows', None)


import sys
sys.path.append('../../TaskExecutionTimeMining/')
from drbart_parser import *
from event_log_transformer import *

#sys.path.append('../../Evaluation')
sys.path.append('../../Evaluation/')
from conduct_evaluation import ConductEvaluation
from advanced_evaluation.advanced_evaluation import SampleOutcomesAdvanced

get_pscores = lambda likelihoods : [pscore(likelihoods[1][i], likelihoods[2][k][3]).compute()[0] for i, k in enumerate(list(likelihoods[0].keys()))]

In [2]:
#model_name = 'bpic_2017_all_2'
model_name = 'helpdesk'

n_processes = 32

log_name = 'test'

with open('../transformed_event_logs/Helpdesk_'+log_name+'.pickle', 'rb') as f:
    test_event_log = pickle.load(f)


test_event_log['Case ID'] = test_event_log['Case ID'].astype(str)
test_event_log['case:concept:name'] = test_event_log['Case ID']
test_event_log['time:timestamp_start'] = test_event_log['Complete Timestamp_start']
test_event_log['time:timestamp_complete'] = test_event_log['Complete Timestamp_complete']


known_resources = ['Value 1', 'Value 10', 'Value 11', 'Value 12', 'Value 13', 'Value 14', 'Value 15', 'Value 16', 'Value 17', 'Value 18', 'Value 19', 'Value 2', 'Value 20', 'Value 21', 'Value 22', 'Value 3', 'Value 4', 'Value 5', 'Value 6', 'Value 7', 'Value 8', 'Value 9']
known_activities = ['Assign seriousness', 'Closed', 'Create SW anomaly', 'DUPLICATE', 'INVALID', 'Insert ticket', 'RESOLVED', 'Require upgrade', 'Resolve SW anomaly', 'Resolve ticket', 'Schedule intervention', 'Take in charge ticket', 'VERIFIED', 'Wait']

In [3]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : '',
                                                        'resources' : False,
                                                        'categorical_args' : ['concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:25<00:00, 35.93it/s]


In [4]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.168965597514896300724067754')

In [5]:
np.mean(get_pscores(likelihoods_A))

np.float64(707705.0900734228)

In [6]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                    
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:18<00:00, 48.90it/s]


In [7]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.193495184001756664922917773')

In [8]:
np.mean(get_pscores(likelihoods_A))

np.float64(746530.6044748863)

In [9]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:16<00:00, 57.29it/s]


In [10]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.626025754096426642617887137')

In [11]:
np.mean(get_pscores(likelihoods_A))

np.float64(975046.7050763957)

In [12]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week'],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:10<00:00, 85.62it/s] 


In [13]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.650801452757366571129317235')

In [14]:
np.mean(get_pscores(likelihoods_A))

np.float64(995159.6712804086)

In [15]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count_resource_count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:10<00:00, 87.11it/s] 


In [16]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.885549887528515098695630032')

In [17]:
np.mean(get_pscores(likelihoods_A))

np.float64(1305806.8921104856)

In [18]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_activity-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:10<00:00, 91.22it/s]


In [19]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.670032851178492673085548556')

In [20]:
np.mean(get_pscores(likelihoods_A))

np.float64(1093786.7190902198)

In [21]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_resource-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:10<00:00, 88.26it/s] 


In [22]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.856029536794339666158972456')

In [23]:
np.mean(get_pscores(likelihoods_A))

np.float64(1229481.1543036834)

In [24]:
drbart_model_path = '../../../models/advanced/'+model_name+'/concept-name_resource_seconds-in-day_day-of-week_activity-count_resoure-count/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource', 'concept_name', 'day_of_week',
                                                                             '(lambda activity_count, known_activities : [0 if activity not in activity_count else activity_count[activity] for activity in known_activities])(activity_count, self.known_activities)',
                                                                              '(lambda resource_count, known_resources : [0 if resource not in resource_count else resource_count[resource] for resource in known_resources])(resource_count, self.known_resources)'
                                                        ],
                                                        'continuous_args' : ['seconds_in_day'],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:09<00:00, 92.48it/s] 


In [25]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.912472610381524610045800164')

In [26]:
np.mean(get_pscores(likelihoods_A))

np.float64(1316629.4403655143)

In [27]:
drbart_model_path = '../../../models/advanced/'+model_name+'/resource/'
evaluator_A = ConductEvaluation(drbart_model_path, SampleOutcomesAdvanced, {
                                                        'activity_key' : 'Activity_start',
                                                        'resource_key' : 'Resource_start',
                                                        'categorical_args' : ['resource'],
                                                        'continuous_args' : [],
                                                        'known_activities' : known_activities,
                                                        'known_resources' : known_resources,
                                                        'strict_parsing' : False
                                                    },
                                    
                                     test_event_log, n_processes=n_processes,
                                )
likelihoods_A = evaluator_A.sample_cases(False, True)

100%|██████████| 919/919 [00:10<00:00, 87.85it/s] 


In [28]:
np.mean([v.ln() for v in likelihoods_A[0].values()])

Decimal('-4.502211526634073577303223437')

In [29]:
np.mean(get_pscores(likelihoods_A))

np.float64(958270.8010142451)